# Module 5.10 — Audit Your Model

**Module 5: Real Project — ML for Animal Adoption | Week 2 | Lab (Session 5.10)**

---

The lecture in 5.9 showed how group-aware cross-validation catches `rescuer_speed`-style leakage on PetFinder. This lab applies the same audit pattern to your own MLP, then takes the audited model the last mile and applies it to AAC dogs.

By the end you'll have:

- A model that doesn't lean on `rescuer_speed`
- An honest score from group-aware cross-validation
- A working application of your model to AAC dogs (with the right caveats)


In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (train_test_split, cross_val_score,
                                       KFold, GroupKFold)
from sklearn.preprocessing import StandardScaler

## Setup — load the data and the same overlap features

Same setup as 5.4. The model we're auditing is the MLP you saved at the end of Lab 2 (the one *without* `rescuer_speed`).


In [ ]:
DATA_URL = 'CONFIGURE_ME'         # ← your instructor will give you this Google Drive URL
AAC_URL  = 'CONFIGURE_ME_AAC'     # ← and this one for the AAC bundle
DATA_DIR = '/content/petfinder'
AAC_DIR  = '/content/aac-mini'

# PetFinder mini.
if not os.path.exists(f'{DATA_DIR}/train.csv'):
    if DATA_URL == 'CONFIGURE_ME':
        raise RuntimeError("Set DATA_URL above to your instructor's URL.")
    os.makedirs(DATA_DIR, exist_ok=True)
    zip_path = f'{DATA_DIR}/petfinder-mini.zip'
    import gdown
    gdown.download(url=DATA_URL, output=zip_path, quiet=False, fuzzy=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)

# AAC mini (Task 4 needs it).
if not os.path.exists(f'{AAC_DIR}/aac_prep.py'):
    if AAC_URL == 'CONFIGURE_ME_AAC':
        raise RuntimeError("Set AAC_URL above to your instructor's URL.")
    os.makedirs(AAC_DIR, exist_ok=True)
    zip_path = f'{AAC_DIR}/aac-mini.zip'
    import gdown
    gdown.download(url=AAC_URL, output=zip_path, quiet=False, fuzzy=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(AAC_DIR)
    os.remove(zip_path)
import sys; sys.path.insert(0, AAC_DIR)

df = pd.read_csv(f'{DATA_DIR}/train.csv')
dogs = df[df['Type'] == 1].copy()
dogs['adopted_in_30_days'] = (dogs['AdoptionSpeed'] <= 2).astype(int)


def to_overlap(d, source='petfinder', tier=5):
    """Map either dataset to a deployable overlap set.

    Tier 5: original five features (age_adult, size, gender_male,
            gender_female, sterilized_yes).
    Tier 6: + is_mixed_breed.
    Tier 7: + is_young.

    AAC doesn't have a MaturitySize equivalent, so `size` defaults to 2
    (medium) when source='aac'. The prediction distribution still shifts —
    that's the lesson.
    """
    out = pd.DataFrame()
    if source == 'petfinder':
        out['age_adult']      = (d['Age'] >= 12).astype(int)
        out['size']           = d['MaturitySize'].clip(1, 3)
        out['gender_male']    = (d['Gender'] == 1).astype(int)
        out['gender_female']  = (d['Gender'] == 2).astype(int)
        out['sterilized_yes'] = (d['Sterilized'] == 1).astype(int)
        if tier >= 6:
            out['is_mixed_breed'] = ((d['Breed1'] == 307) | (d['Breed2'] != 0)).astype(int)
        if tier >= 7:
            out['is_young']       = (d['Age'] < 6).astype(int)
    elif source == 'aac':
        out['age_adult']      = d['age_adult'].astype(int)
        out['size']           = 2  # AAC lacks size; default to medium
        out['gender_male']    = d['gender_male'].astype(int)
        out['gender_female']  = d['gender_female'].astype(int)
        out['sterilized_yes'] = d['sterilized_yes'].astype(int)
        if tier >= 6:
            out['is_mixed_breed'] = d['is_mixed_breed'].astype(int)
        if tier >= 7:
            out['is_young']       = d['is_young'].astype(int)
    return out


---

### Task 1: Reproduce the lecture's gap

The lecture showed that random CV vs group-aware CV exposes the leak. Reproduce it on your own setup. We'll use logistic regression here for speed — it's fast enough that 5-fold CV runs in a few seconds, and the leakage gap is independent of model class.

```python
# Build the leaky feature first (same as Lab 2 Task 4).
rescuer_speed_full = dogs.groupby('RescuerID')['adopted_in_30_days'].mean()
X_leaky = np.column_stack([X_pf_s, dogs['RescuerID'].map(rescuer_speed_full).values.reshape(-1, 1)])

random_cv = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_leaky, y_pf,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
)
group_cv = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_leaky, y_pf,
    cv=GroupKFold(n_splits=5), groups=groups,
)

print(f'Random CV:      {random_cv.mean():.3f} ± {random_cv.std():.3f}')
print(f'Group-aware CV: {group_cv.mean():.3f} ± {group_cv.std():.3f}')
print(f'Gap (leakage):  {random_cv.mean() - group_cv.mean():+.3f}')
```


In [ ]:
# Task 1: your code here


---

### Task 2: Drop `rescuer_speed`, run group-aware CV again

Most of the gap was coming from that one feature. Re-run group-aware CV on the *clean* overlap-feature set (without `rescuer_speed`). Compare to Task 1's group_cv mean.

```python
clean_cv = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_pf_s, y_pf,
    cv=GroupKFold(n_splits=5), groups=groups,
)
print(f'Clean overlap features (group CV): {clean_cv.mean():.3f}')
```

What you should see: dropping `rescuer_speed` knocks down the random-CV number, but the group-aware-CV number is similar. That's the audit working — you've removed a feature that was inflating the random-split score without actually predicting better on new shelters.


In [ ]:
# Task 2: your code here


---

### Task 3: Audit your MLP

LR is fast for cross-validation but your real model is the MLP from Lab 2. Run a single-split MLP audit: train it on a 75/25 split where train and test share *no* rescuers, evaluate, and report.

```python
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X_pf_s, y_pf, groups=groups))

X_tr, X_te = X_pf_s[train_idx], X_pf_s[test_idx]
y_tr, y_te = y_pf[train_idx], y_pf[test_idx]

class TabularMLP(nn.Module):
    def __init__(self, n_features, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
mlp = TabularMLP(X_tr.shape[1]).to(device)
opt = optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
Xtr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
ytr_t = torch.tensor(y_tr, dtype=torch.float32, device=device)
Xte_t = torch.tensor(X_te, dtype=torch.float32, device=device)
for _ in range(150):
    mlp.train(); opt.zero_grad()
    loss_fn(mlp(Xtr_t), ytr_t).backward(); opt.step()

mlp.eval()
with torch.no_grad():
    preds = (torch.sigmoid(mlp(Xte_t)) > 0.5).int().cpu().numpy()
audited_acc = float((preds == y_te).mean())
print(f'Audited MLP (group-disjoint test): {audited_acc:.3f}')
```

That's the number you'd actually defend in your brief. Don't expect it to beat the LR-based CV number; the lesson here isn't "MLP wins," it's "the audited number is smaller and more honest than the random-CV one."


In [ ]:
# Task 3: your code here


---

### Task 4: Apply your audited MLP to AAC

You've done the work on PetFinder. Now use it on a different shelter and see what changes.

Austin Animal Center (AAC) is a real US city shelter — about 40,000 cleaned records, dogs and cats, no photos or descriptions. Pure tabular cross-dataset data. Load it, map it to the same 5 features your MLP was trained on, score the dogs, and look at the prediction distribution.

The crucial note: AAC doesn't have a `MaturitySize` equivalent, so we default `size=2` (medium) for every AAC dog. That's an approximation — your predictions on AAC are still informative as a *distribution* of ranks, but per-animal predictions inherit that approximation.

```python
from aac_prep import load_aac
df_aac = load_aac()
aac_dogs = df_aac[df_aac['Type'] == 1].copy()

X_aac = to_overlap(aac_dogs, source='aac').values.astype(float)
X_aac_s = scaler.transform(X_aac)

mlp.eval()
with torch.no_grad():
    aac_logits = mlp(torch.tensor(X_aac_s, dtype=torch.float32, device=device))
    aac_proba = torch.sigmoid(aac_logits).cpu().numpy()

print(f'AAC dogs scored: {len(aac_proba):,}')
print(f'  min:    {aac_proba.min():.3f}')
print(f'  median: {np.median(aac_proba):.3f}')
print(f'  max:    {aac_proba.max():.3f}')
print(f'  mean:   {aac_proba.mean():.3f}')

# Compare to PetFinder test-set predictions
pf_proba = ... # from Task 3's audited MLP, scored on the test split

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].hist(pf_proba, bins=30, color='#5BC0DE', edgecolor='white')
axes[0].axvline(0.5, color='black', linestyle='--', label='Decision threshold')
axes[0].set_xlabel('Predicted P(adopted in 30 days)')
axes[0].set_ylabel('count')
axes[0].set_title('PetFinder test dogs')
axes[0].legend()

axes[1].hist(aac_proba, bins=30, color='#F0AD4E', edgecolor='white')
axes[1].axvline(0.5, color='black', linestyle='--')
axes[1].set_xlabel('Predicted P(adopted in 30 days)')
axes[1].set_title('AAC dogs (PetFinder-trained MLP)')
plt.tight_layout(); plt.show()
```

Notice how the AAC distribution differs from PetFinder's. The model's predictions on AAC dogs cluster in a different band — and that's not because AAC dogs are actually adopted at those rates (the actual AAC 30-day rate is ~34%, very different from PetFinder's 46%). The model is using PetFinder's features and weights, applied to a population that looks different. *That's distribution shift in action.*

AAC has one big advantage over the PetFinder data: **AAC publishes actual outcomes**, so you can check whether your model's rankings actually agree with the order AAC dogs got adopted. M06's Taiwan deployment lab does exactly that on a shelter where outcomes are *not* published.


In [ ]:
# Task 4: your code here


---

## Writeup

In a markdown cell, write 5-7 bullets answering:

- What was the random-CV vs group-CV gap when `rescuer_speed` was included?
- After dropping `rescuer_speed`, what's the audited score from Task 3?
- If a stranger ran your audited model on a new shelter's data, what would you expect to happen?
- What number will go in your brief in M06?
- For the AAC application: what was the prediction distribution? How did it differ from PetFinder? What does that tell you about applying a model across shelters?
- One sentence: what would AAC (or any second shelter) need to publish for you to *validate* your model's predictions there, not just rank them?

---

## Summary

- Reproduced the lecture's audit on the model from Lab 2.
- Removed `rescuer_speed`, ran group-aware CV, found the honest score.
- Trained an audited MLP on a group-disjoint split — that's your defendable number.
- Applied the audited MLP to AAC and saw the prediction distribution shift — distribution shift in action.

---

**Next session:** Where data comes from, where the ethical lines are, and what publication policy itself can hide.
